In [2]:
# whoami
import sys
print(sys.executable)
print(sys.version)

/opt/homebrew/Cellar/jupyterlab/4.6.2/libexec/bin/python
3.14.7 (main, Aug  5 2026, 10:29:49) [Clang 21.0.0 (clang-2100.1.1.101)]


In [3]:
%pip install pandas matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 8.8 MB/s  0:00:01 8.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 16.8 MB/s  0:00:007.0 MB/s eta 0:00:01:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 12.7 MB/s  0:00:007.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 22.2 MB/s  0:00:00a 0:00:01
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [seaborn]━━━ 7/9 [matplotlib]as]
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Obligatory imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Begin data description and first look

In [21]:
DATA_DIR = Path("data")
datasets = {}
csv_files = sorted(DATA_DIR.glob("*.csv"))

for file in csv_files:
    datasets[file.stem] = pd.read_csv(
        file,
        low_memory=False,
        keep_default_na=True,
        na_values=["", " ", "NA", "N/A", "NULL", "null", "None"],
    )

    print(f"File Name: {file.name}")
    print(f"File Shape: {datasets[file.stem].shape}")
    print(f"File Columns: \n{datasets[file.stem].columns}")
    print(f"Exact duplicate rows: {datasets[file.stem].duplicated().sum()}")

    summary = pd.DataFrame({
        "dtype": datasets[file.stem].dtypes.astype(str),
        "non_null": datasets[file.stem].notna().sum(),
        "missing": datasets[file.stem].isna().sum(),
        "missing_pct": (datasets[file.stem].isna().mean() * 100).round(2),
        "unique": datasets[file.stem].nunique(dropna=True),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])

    display(summary)

File Name: listings.csv
File Shape: (10242, 90)
File Columns: 
Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_profile_id', 'host_profile_url', 'host_name',
       'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months',
       'hosts_time_as_host_years', 'hosts_time_as_host_months',
       'host_location', 'host_about', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood',
       'host_listings_count', 'host_total_listings_count',
       'host_verifications', 'host_has_profile_pic', 'host_identity_verified',
       'neighbourhood', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'ameniti

,dtype,non_null,missing,missing_pct,unique
neighborhood_overview,float64,0,10242,100.0,0
host_since,float64,0,10242,100.0,0
host_response_time,float64,0,10242,100.0,0
host_response_rate,float64,0,10242,100.0,0
host_acceptance_rate,float64,0,10242,100.0,0
...,...,...,...,...,...
host_is_superhost,str,10242,0,0.0,2
host_has_profile_pic,str,10242,0,0.0,2
host_identity_verified,str,10242,0,0.0,2
calendar_last_scraped,str,10242,0,0.0,2


File Name: reviews.csv
File Shape: (838603, 6)
File Columns: 
Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='str')
Exact duplicate rows: 0


,dtype,non_null,missing,missing_pct,unique
comments,str,838314,289,0.03,801327
id,int64,838603,0,0.00,838603
reviewer_id,int64,838603,0,0.00,750529
reviewer_name,str,838602,1,0.00,63115
listing_id,int64,838603,0,0.00,8945
date,str,838603,0,0.00,5200


In [22]:
# copy known csvs
listings = datasets["listings"].copy()
reviews = datasets["reviews"].copy()

In [27]:
print("Listing ID unique:", listings["id"].is_unique)
print("Review ID unique:", reviews["id"].is_unique)

print("Duplicate listing IDs:", listings["id"].duplicated().sum())
print("Duplicate review IDs:", reviews["id"].duplicated().sum())
print("Missing review listing IDs:", reviews["listing_id"].isna().sum())

Listing ID unique: True
Review ID unique: True
Duplicate listing IDs: 0
Duplicate review IDs: 0
Missing review listing IDs: 0


In [24]:
orphan_review_mask = ~reviews["listing_id"].isin(listings["id"])

print("Orphan review rows:", orphan_review_mask.sum())
print(
    "Orphan listing IDs:",
    reviews.loc[orphan_review_mask, "listing_id"].nunique()
)

listings_without_reviews = ~listings["id"].isin(reviews["listing_id"])

print("Listings without reviews:", listings_without_reviews.sum())
print(
    "Percent of listings with reviews:",
    (~listings_without_reviews).mean() * 100
)

Orphan review rows: 740
Orphan listing IDs: 9
Listings without reviews: 1306
Percent of listings with reviews: 87.24858426088655


In [25]:
def column_quality(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
        "unique": df.nunique(dropna=True),
    }).sort_values(
        ["missing_pct", "unique"],
        ascending=[False, True],
    )

listing_quality = column_quality(listings)

all_missing_columns = listing_quality.query(
    "missing_pct == 100"
).index.tolist()

mostly_missing_columns = listing_quality.query(
    "90 <= missing_pct < 100"
).index.tolist()

constant_columns = listing_quality.query(
    "unique <= 1 and missing_pct < 100"
).index.tolist()

print("All missing:", all_missing_columns)
print("At least 90% missing:", mostly_missing_columns)
print("Constant columns:", constant_columns)

All missing: ['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'instant_bookable']
At least 90% missing: []
Constant columns: ['scrape_id']


## drop empty columns

In [26]:
listings_clean = listings.drop(columns=all_missing_columns).copy()
reviews_clean = reviews.dropna(how="all", axis="columns").copy()

## clean text fields

In [28]:
def strip_text_columns(df):
    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

    return df

listings_clean = strip_text_columns(listings_clean)
reviews_clean = strip_text_columns(reviews_clean)

## clean date fields

In [29]:
listing_date_columns = [
    "last_scraped",
    "host_since",
    "price_quote_checkin_date",
    "price_quote_checkout_date",
    "calendar_last_scraped",
    "first_review",
    "last_review",
]

for column in listing_date_columns:
    if column in listings_clean.columns:
        listings_clean[column] = pd.to_datetime(
            listings_clean[column],
            errors="coerce",
        )

reviews_clean["date"] = pd.to_datetime(
    reviews_clean["date"],
    errors="coerce",
)

## clean money fields

In [31]:
currency_columns = [
    "price",
    "price_quote_total_price",
    "price_quote_price_per_night",
    "estimated_revenue_l365d",
]

for column in currency_columns:
    if column in listings_clean.columns:
        original = listings_clean[column].copy()

        listings_clean[column] = pd.to_numeric(
            original
            .astype("string")
            .str.replace(r"[$,]", "", regex=True),
            errors="coerce",
        )

## clean percentages

In [32]:
def parse_percentage(series):
    text = series.astype("string").str.strip()
    has_percent_sign = text.str.endswith("%", na=False)

    values = pd.to_numeric(
        text.str.rstrip("%"),
        errors="coerce",
    )

    values.loc[has_percent_sign] /= 100
    return values

percentage_columns = [
    "host_response_rate",
    "host_acceptance_rate",
]

for column in percentage_columns:
    if column in listings_clean.columns:
        listings_clean[column] = parse_percentage(
            listings_clean[column]
        )

## clean booleans

In [33]:
boolean_columns = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable",
]

boolean_mapping = {
    "t": True,
    "true": True,
    "yes": True,
    "1": True,
    "f": False,
    "false": False,
    "no": False,
    "0": False,
}

for column in boolean_columns:
    if column in listings_clean.columns:
        listings_clean[column] = (
            listings_clean[column]
            .astype("string")
            .str.lower()
            .map(boolean_mapping)
            .astype("boolean")
        )

## aggregate reviews

In [35]:
review_summary = (
    reviews_clean
    .groupby("listing_id", as_index=False)
    .agg(
        reviews_in_file=("id", "size"),
        unique_reviewers=("reviewer_id", "nunique"),
        first_review_in_file=("date", "min"),
        last_review_in_file=("date", "max"),
        comments_present=("comments", "count"),
    )
)

review_summary["comments_missing"] = (
    review_summary["reviews_in_file"]
    - review_summary["comments_present"]
)

display(review_summary.head())

,listing_id,reviews_in_file,unique_reviewers,first_review_in_file,last_review_in_file,comments_present,comments_missing
0,6422,667,645,2009-04-30,2020-03-03,667,0
1,39870,638,585,2016-09-16,2026-06-21,638,0
2,72906,801,793,2011-06-09,2026-06-21,801,0
3,258817,97,97,2011-12-12,2023-12-27,97,0
4,289242,76,76,2011-12-28,2019-01-12,76,0


# Combine the files

In [36]:
listing_level = listings_clean.merge(
    review_summary,
    left_on="id",
    right_on="listing_id",
    how="left",
    validate="one_to_one",
    indicator="review_match",
)

listing_level = listing_level.drop(columns="listing_id")

count_columns = [
    "reviews_in_file",
    "unique_reviewers",
    "comments_present",
    "comments_missing",
]

listing_level[count_columns] = (
    listing_level[count_columns]
    .fillna(0)
    .astype("int64")
)

display(listing_level["review_match"].value_counts())
print("Final shape:", listing_level.shape)

review_match
both          8936
left_only     1306
right_only       0
Name: count, dtype: int64

Final shape: (10242, 84)


## Save the results

In [38]:
output_file = DATA_DIR / "listings_with_review_summary.csv"

listing_level.drop(columns="review_match").to_csv(
    output_file,
    index=False,
)

print(output_file)

data/listings_with_review_summary.csv
